# Animation: Gradient Descent

Gradient descent is the engine behind most ML model training.

The idea: **follow the steepest downhill direction** of the loss function, one small step at a time.

This notebook shows:
1. The loss surface as a 2D contour map
2. The path the optimizer takes from a random start to the minimum
3. How the **learning rate** changes the behavior

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

np.random.seed(0)

# ── 2D loss surface: elongated bowl (simulates correlated features) ──
def loss(w1, w2):
    return 0.5 * w1**2 + 3.0 * w2**2 + 0.8 * w1 * w2

def grad(w1, w2):
    dw1 = w1 + 0.4 * w2
    dw2 = 6.0 * w2 + 0.4 * w1
    return np.array([dw1, dw2])

# ── Run gradient descent for three learning rates ──────────────────
def run_gd(lr, n_steps=60, start=None):
    if start is None:
        start = np.array([-3.5, 2.5])
    w = start.copy().astype(float)
    path = [w.copy()]
    for _ in range(n_steps):
        g = grad(w[0], w[1])
        w -= lr * g
        path.append(w.copy())
        if np.linalg.norm(g) < 1e-6:
            break
    return np.array(path)

start = np.array([-3.5, 2.5])
paths = {
    'Good (lr=0.08)':  run_gd(0.08, start=start),
    'Slow  (lr=0.01)': run_gd(0.01, start=start),
    'Fast  (lr=0.19)': run_gd(0.19, start=start, n_steps=25),
}

print('Path lengths:', {k: len(v) for k, v in paths.items()})

In [ ]:
# ── Animation: good learning rate (the default teaching example) ───
path = paths['Good (lr=0.08)']
N = len(path)

w1_range = np.linspace(-4.5, 4.5, 300)
w2_range = np.linspace(-3.5, 3.5, 300)
W1, W2 = np.meshgrid(w1_range, w2_range)
L = loss(W1, W2)

fig, ax = plt.subplots(figsize=(7, 6))
levels = np.logspace(-1, 2, 20)
ax.contourf(W1, W2, L, levels=levels, cmap='YlOrRd_r', alpha=0.7)
cs = ax.contour(W1, W2, L, levels=levels, colors='gray', linewidths=0.4, alpha=0.5)
ax.set_xlabel('Weight w₁', fontsize=12)
ax.set_ylabel('Weight w₂', fontsize=12)
ax.set_title('Gradient Descent on the Loss Surface', fontsize=13)

# Mark start and minimum
ax.scatter(*start, color='#3498db', s=120, zorder=5, label='Start')
ax.scatter(0, 0, color='#2ecc71', s=200, marker='*', zorder=5, label='Minimum')
ax.legend(fontsize=10, loc='upper right')

# Line + current point placeholders
line,  = ax.plot([], [], 'o-', color='#2c3e50', lw=1.8, ms=3, zorder=4)
point, = ax.plot([], [], 'o', color='#e74c3c', ms=10, zorder=6)
info   = ax.text(0.02, 0.04, '', transform=ax.transAxes, fontsize=10,
                 va='bottom',
                 bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.85))

def update(frame):
    pts = path[:frame + 1]
    line.set_data(pts[:, 0], pts[:, 1])
    point.set_data([pts[-1, 0]], [pts[-1, 1]])
    current_loss = loss(pts[-1, 0], pts[-1, 1])
    info.set_text(f'Step {frame}  |  Loss = {current_loss:.4f}')
    return line, point, info

anim = FuncAnimation(fig, update, frames=N,
                     interval=120, repeat=False, blit=True)
plt.tight_layout()
plt.close()
HTML(anim.to_jshtml())

In [ ]:
# ── Static comparison: three learning rates side by side ───────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

lr_labels = [
    ('Slow  (lr=0.01)', '#3498db',  'Too slow — many steps needed'),
    ('Good (lr=0.08)',  '#2ecc71',  'Just right — converges smoothly'),
    ('Fast  (lr=0.19)', '#e74c3c',  'Too fast — oscillates!'),
]

for ax, (label, color, desc) in zip(axes, lr_labels):
    p = paths[label]
    ax.contourf(W1, W2, L, levels=levels, cmap='YlOrRd_r', alpha=0.65)
    ax.contour(W1, W2, L, levels=levels, colors='gray', linewidths=0.3, alpha=0.4)
    ax.plot(p[:, 0], p[:, 1], 'o-', color=color, lw=1.8, ms=4, alpha=0.9)
    ax.scatter(*start, color='#2c3e50', s=100, zorder=5)
    ax.scatter(0, 0, color='gold', s=200, marker='*', zorder=5)
    ax.set_title(f'{label}\n{desc}', fontsize=11)
    ax.set_xlabel('w₁')
    ax.set_ylabel('w₂')

plt.suptitle('Effect of Learning Rate on Gradient Descent', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## What to notice

- **The path spirals inward** because the loss surface is elongated — the gradient is steeper in one direction than the other
- **Too small a learning rate:** the optimizer is painfully slow — hundreds of steps to converge
- **Too large a learning rate:** the steps overshoot — the optimizer bounces back and forth, or diverges entirely
- **Just right:** smooth convergence in a reasonable number of steps

**In practice:** Start with a learning rate around `0.01` or `0.001` and use a learning rate scheduler to reduce it over time. Adam optimizer adapts the learning rate automatically — that's why it's the default in deep learning.